# 🎨 StyleAligned — a set of images in one shared style

Generate several images that **share one style** (training-free) — pick a style and a few subjects.

**Setup:** Runtime → GPU (A100) · free [HF token](https://huggingface.co/settings/tokens) · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).

### 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

### 2 · Sign in

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"   # avoid transient HF-hub read timeouts on big downloads
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()

### 3 · Load the model (~2–3 min first run)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/style-aligned-flux-modular", trust_remote_code=True)
pipe.load_components(dtype=torch.bfloat16); pipe.to("cuda")
print("✅ ready:", type(pipe.blocks).__name__)  # StyleAlignedFluxBlock

### 4 · Pick a style + subjects → a style-consistent set
Every image shares the style; the first is the anchor. Edit `STYLE` and `SUBJECTS`, then run.

In [ ]:
#@title Generate a styled set { display-mode: "form" }
STYLE = "flat vector sticker art"  #@param {type:"string"}
SUBJECTS = "a red panda, a fox, a rabbit"  #@param {type:"string"}
seed = 0  #@param {type:"integer"}
import torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
subs=[s.strip() for s in SUBJECTS.split(",") if s.strip()]
prompts=[f"{s}, {STYLE}" for s in subs]
g=torch.Generator("cuda").manual_seed(int(seed))
imgs=pipe(prompts=prompts, height=1024, width=1024, num_inference_steps=28, generator=g).images
S=460; W=len(imgs)*S+(len(imgs)+1)*8; row=Image.new("RGB",(W,S+34),"white"); d=ImageDraw.Draw(row)
try: F=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",22)
except: F=ImageFont.load_default()
for i,im in enumerate(imgs): x=8+i*(S+8); row.paste(im.resize((S,S)),(x,0)); d.text((x+6,S+6),subs[i][:36],fill="black",font=F)
row.save("result.png"); print(f"one '{STYLE}' style, distinct subjects:"); display(row.resize((min(W,1400),int((S+34)*min(W,1400)/W))))

### Download

In [ ]:
from google.colab import files
files.download("result.png")

---
[`remyxai/style-aligned-flux-modular`](https://huggingface.co/remyxai/style-aligned-flux-modular) · training-free · non-commercial.